In [0]:
# Databricks Week 7 Assignment
print("Hello Databricks")

Hello Databricks


In [0]:
# Read customer_master table
df = spark.table("workspace.default.customer_master")

# Display first few records
display(df)

customer_id,name,city,age,email
1,Priya,Surat,35,priya1@gmail.com
2,Sneha,Delhi,26,sneha2@gmail.com
3,Priya,Vadodara,23,priya3@gmail.com
4,Harsh,Indore,20,harsh4@gmail.com
5,Aditya,Pune,31,aditya5@gmail.com
6,Sneha,Vadodara,56,sneha6@gmail.com
7,Aditya,Vadodara,30,aditya7@gmail.com
8,Meera,Indore,32,meera8@gmail.com
9,Arjun,Rajkot,35,arjun9@gmail.com
10,Aditya,Mumbai,45,aditya10@gmail.com


In [0]:
# Number of records
print("Total Records:", df.count())

Total Records: 10014


In [0]:
df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- email: string (nullable = true)



In [0]:
# NULL Values..
from pyspark.sql.functions import col

df.filter(col("name").isNull() | (col("name") == "")).show()

+-----------+----+--------+---+--------------------+
|customer_id|name|    city|age|               email|
+-----------+----+--------+---+--------------------+
|       1000|NULL|  Jaipur| 48| kriti1000@gmail.com|
|       2000|NULL|  Mumbai| 23|anjali2000@gmail.com|
|       3000|NULL|Vadodara| -5| priya3000@gmail.com|
|       4000|NULL|  Rajkot| 59|  neha4000@gmail.com|
|       5000|NULL|  Mumbai| 51|  riya5000@gmail.com|
|       6000|NULL|  Jaipur| -5|  neha6000@gmail.com|
|       7000|NULL|   Surat| 47| meera7000@gmail.com|
|       7000|NULL|   Surat| 47| meera7000@gmail.com|
|       8000|NULL|   Delhi| 36| kriti8000@gmail.com|
|       9000|NULL|    Pune| -5| kavya9000@gmail.com|
|      10000|NULL|    Pune| 24|nisha10000@gmail.com|
+-----------+----+--------+---+--------------------+



In [0]:
# Duplicate Records
from pyspark.sql.functions import count

df.groupBy("customer_id").count().filter(col("count") > 1).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|        700|    2|
|       1400|    2|
|       2100|    2|
|       2800|    2|
|       3500|    2|
|       4200|    2|
|       4900|    2|
|       5600|    2|
|       6300|    2|
|       7000|    2|
|       7700|    2|
|       8400|    2|
|       9100|    2|
|       9800|    2|
+-----------+-----+



In [0]:
# Remove NULL and Duplicates..
clean_df = df.dropna().dropDuplicates()

In [0]:
print("Records after cleaning:", clean_df.count())

Records after cleaning: 9990


In [0]:
# Save cleaned data as Delta Table
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.customer_master_clean")

In [0]:
delta_df = spark.table("workspace.default.customer_master_clean")

display(delta_df)

customer_id,name,city,age,email
20,Sneha,Ahmedabad,23,sneha20@gmail.com
93,Pooja,Delhi,21,pooja93@gmail.com
138,Nisha,Surat,29,nisha138@gmail.com
144,Sneha,Delhi,19,sneha144@gmail.com
145,Mohit,Indore,39,mohit145@gmail.com
191,Aman,Vadodara,49,aman191@gmail.com
202,Harsh,Indore,48,harsh202@gmail.com
211,Aditya,Indore,55,aditya211@gmail.com
230,Pooja,Rajkot,42,pooja230@gmail.com
232,Harsh,Delhi,24,harsh232@gmail.com


In [0]:
print("Records in Delta Table:", delta_df.count())

Records in Delta Table: 9990


In [0]:
from pyspark.sql.functions import col, lit

# Existing customers (will be UPDATED)
updated_df = delta_df.filter(col("customer_id") <= 20) \
    .withColumn("city", lit("Bangalore")) \
    .withColumn("age", col("age") + 2)

# New customers (will be INSERTED)
new_customers = [
    (10001, "Aditya", "Surat", 22, "aditya10001@gmail.com"),
    (10002, "Rahul", "Pune", 24, "rahul10002@gmail.com"),
    (10003, "Neha", "Mumbai", 27, "neha10003@gmail.com"),
    (10004, "Karan", "Delhi", 30, "karan10004@gmail.com"),
    (10005, "Pooja", "Ahmedabad", 25, "pooja10005@gmail.com")
]

new_df = spark.createDataFrame(
    new_customers,
    ["customer_id", "name", "city", "age", "email"]
)

# Combine both datasets
incremental_df = updated_df.union(new_df)

display(incremental_df)

customer_id,name,city,age,email
20,Sneha,Bangalore,25,sneha20@gmail.com
5,Aditya,Bangalore,33,aditya5@gmail.com
2,Sneha,Bangalore,28,sneha2@gmail.com
6,Sneha,Bangalore,58,sneha6@gmail.com
13,Neha,Bangalore,26,neha13@gmail.com
1,Priya,Bangalore,37,priya1@gmail.com
3,Priya,Bangalore,25,priya3@gmail.com
15,Aman,Bangalore,49,aman15@gmail.com
9,Arjun,Bangalore,37,arjun9@gmail.com
17,Neha,Bangalore,38,neha17@gmail.com


In [0]:
incremental_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.customer_incremental")

In [0]:
inc_df = spark.table("workspace.default.customer_incremental")

display(inc_df)

customer_id,name,city,age,email
20,Sneha,Bangalore,25,sneha20@gmail.com
5,Aditya,Bangalore,33,aditya5@gmail.com
2,Sneha,Bangalore,28,sneha2@gmail.com
6,Sneha,Bangalore,58,sneha6@gmail.com
13,Neha,Bangalore,26,neha13@gmail.com
1,Priya,Bangalore,37,priya1@gmail.com
3,Priya,Bangalore,25,priya3@gmail.com
15,Aman,Bangalore,49,aman15@gmail.com
9,Arjun,Bangalore,37,arjun9@gmail.com
17,Neha,Bangalore,38,neha17@gmail.com


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "workspace.default.customer_master_clean")

(
    target.alias("target")
    .merge(
        inc_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "name": "source.name",
        "city": "source.city",
        "age": "source.age",
        "email": "source.email"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "city": "source.city",
        "age": "source.age",
        "email": "source.email"
    })
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
final_df = spark.table("workspace.default.customer_master_clean")

display(final_df)

customer_id,name,city,age,email
93,Pooja,Delhi,21,pooja93@gmail.com
138,Nisha,Surat,29,nisha138@gmail.com
144,Sneha,Delhi,19,sneha144@gmail.com
145,Mohit,Indore,39,mohit145@gmail.com
191,Aman,Vadodara,49,aman191@gmail.com
202,Harsh,Indore,48,harsh202@gmail.com
211,Aditya,Indore,55,aditya211@gmail.com
230,Pooja,Rajkot,42,pooja230@gmail.com
232,Harsh,Delhi,24,harsh232@gmail.com
236,Anjali,Pune,50,anjali236@gmail.com


In [0]:
print("Final Record Count:", final_df.count())

Final Record Count: 9995


In [0]:
from pyspark.sql.functions import col

final_df.filter(col("customer_id") <= 20).show()

+-----------+------+---------+---+------------------+
|customer_id|  name|     city|age|             email|
+-----------+------+---------+---+------------------+
|         20| Sneha|Bangalore| 25| sneha20@gmail.com|
|          5|Aditya|Bangalore| 33| aditya5@gmail.com|
|          2| Sneha|Bangalore| 28|  sneha2@gmail.com|
|          6| Sneha|Bangalore| 58|  sneha6@gmail.com|
|         13|  Neha|Bangalore| 26|  neha13@gmail.com|
|          1| Priya|Bangalore| 37|  priya1@gmail.com|
|          3| Priya|Bangalore| 25|  priya3@gmail.com|
|         15|  Aman|Bangalore| 49|  aman15@gmail.com|
|          9| Arjun|Bangalore| 37|  arjun9@gmail.com|
|         17|  Neha|Bangalore| 38|  neha17@gmail.com|
|          7|Aditya|Bangalore| 32| aditya7@gmail.com|
|         10|Aditya|Bangalore| 47|aditya10@gmail.com|
|         11| Rohit|Bangalore| 29| rohit11@gmail.com|
|         16| Meera|Bangalore| 44| meera16@gmail.com|
|          4| Harsh|Bangalore| 22|  harsh4@gmail.com|
|          8| Meera|Bangalor

In [0]:
final_df.filter(col("customer_id") >= 10001).show()

+-----------+------+---------+---+--------------------+
|customer_id|  name|     city|age|               email|
+-----------+------+---------+---+--------------------+
|      10001|Aditya|    Surat| 22|aditya10001@gmail...|
|      10002| Rahul|     Pune| 24|rahul10002@gmail.com|
|      10003|  Neha|   Mumbai| 27| neha10003@gmail.com|
|      10004| Karan|    Delhi| 30|karan10004@gmail.com|
|      10005| Pooja|Ahmedabad| 25|pooja10005@gmail.com|
+-----------+------+---------+---+--------------------+



In [0]:
print("="*40)
print("Delta Lake MERGE Assignment Completed")
print("="*40)

print("Original Records :", df.count())
print("After Cleaning   :", clean_df.count())
print("After MERGE      :", final_df.count())

Delta Lake MERGE Assignment Completed
Original Records : 10014
After Cleaning   : 9990
After MERGE      : 9995


In [0]:
# Final DataFrame
display(final_df)

customer_id,name,city,age,email
93,Pooja,Delhi,21,pooja93@gmail.com
138,Nisha,Surat,29,nisha138@gmail.com
144,Sneha,Delhi,19,sneha144@gmail.com
145,Mohit,Indore,39,mohit145@gmail.com
191,Aman,Vadodara,49,aman191@gmail.com
202,Harsh,Indore,48,harsh202@gmail.com
211,Aditya,Indore,55,aditya211@gmail.com
230,Pooja,Rajkot,42,pooja230@gmail.com
232,Harsh,Delhi,24,harsh232@gmail.com
236,Anjali,Pune,50,anjali236@gmail.com
